In [0]:

# ============================================================
# RETAILPULSE - REUSABLE BRONZE INGESTION PIPELINE
# ============================================================


# Step 1: Import required libraries

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid


# ============================================================
# Step 2: Configure source and target locations
# ============================================================

source_folder = "/Volumes/retailpulse/dev/olist_raw/"

target_schema = "retailpulse.bronze"

expected_row_counts = {
    "customers": 99441,
    "orders": 99441,
    "order_items": 112650,
    "order_payments": 103886,
    "order_reviews": 99224,
    "products": 32951,
    "sellers": 3095,
    "geolocation": 1000163,
    "product_category_name_translation": 71
}


# ============================================================
# Step 3: Configure all nine datasets
# ============================================================

datasets = {
    "customers": {
        "source_file": "olist_customers_dataset.csv",
        "target_table": "customers",
        "primary_key": ["customer_id"]
    },

    "orders": {
        "source_file": "olist_orders_dataset.csv",
        "target_table": "orders",
        "primary_key": ["order_id"]
    },

    "order_items": {
        "source_file": "olist_order_items_dataset.csv",
        "target_table": "order_items",
        "primary_key": ["order_id", "order_item_id"]
    },

    "order_payments": {
        "source_file": "olist_order_payments_dataset.csv",
        "target_table": "order_payments",
        "primary_key": ["order_id", "payment_sequential"]
    },

    "order_reviews": {
        "source_file": "olist_order_reviews_dataset.csv",
        "target_table": "order_reviews",
        "primary_key": ["review_id", "order_id"]
    },

    "products": {
        "source_file": "olist_products_dataset.csv",
        "target_table": "products",
        "primary_key": ["product_id"]
    },

    "sellers": {
        "source_file": "olist_sellers_dataset.csv",
        "target_table": "sellers",
        "primary_key": ["seller_id"]
    },

    "geolocation": {
        "source_file": "olist_geolocation_dataset.csv",
        "target_table": "geolocation",
        "primary_key": []
    },

    "product_category_name_translation": {
        "source_file": "product_category_name_translation.csv",
        "target_table": "product_category_name_translation",
        "primary_key": ["product_category_name"]
    }
}


# ============================================================
# Step 4: Create reusable ingestion function
# ============================================================

def ingest_dataset(dataset_name, dataset_config):

    print("\n" + "=" * 60)
    print(f"Starting ingestion for: {dataset_name}")
    print("=" * 60)

    source_file = dataset_config["source_file"]
    primary_key = dataset_config["primary_key"]

    source_path = source_folder + source_file

    target_table = (
        target_schema + "." + dataset_config["target_table"]
    )

    run_id = (
        f"{dataset_name}_"
        f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid.uuid4().hex[:8]}"
    )

    try:

        # ----------------------------------------------------
        # Step 4.1: Read the CSV file
        # ----------------------------------------------------

        if dataset_name == "order_reviews":

            dataframe = (
                spark.read
                .format("csv")
                .option("header", True)
                .option("inferSchema", False)
                .option("multiLine", True)
                .option("quote", '"')
                .option("escape", '"')
                .option(
                    "unescapedQuoteHandling",
                    "STOP_AT_DELIMITER"
                )
                .load(source_path)
            )

        else:

            dataframe = (
                spark.read
                .format("csv")
                .option("header", True)
                .option("inferSchema", False)
                .load(source_path)
            )

        print("Source file loaded successfully")


        # ----------------------------------------------------
        # Step 4.2: Check row count and column count
        # ----------------------------------------------------

        row_count = dataframe.count()
        column_count = len(dataframe.columns)

        print(f"Source rows: {row_count:,}")
        print(f"Source columns: {column_count}")


        # ----------------------------------------------------
        # Step 4.3: Validate expected row count
        # ----------------------------------------------------

        expected_row_count = expected_row_counts.get(
            dataset_name
        )

        if (
            expected_row_count is not None
            and row_count != expected_row_count
        ):
            raise ValueError(
                f"Expected {expected_row_count} rows, "
                f"but found {row_count}"
            )


        # ----------------------------------------------------
        # Step 4.4: Validate primary key columns
        # ----------------------------------------------------

        primary_key_null_count = 0
        primary_key_duplicate_count = 0

        for column in primary_key:

            if column not in dataframe.columns:
                raise ValueError(
                    f"Primary key column not found: {column}"
                )

            null_count = (
                dataframe
                .filter(F.col(column).isNull())
                .count()
            )

            print(
                f"NULL values in {column}: {null_count:,}"
            )

            primary_key_null_count += null_count


        # Check uniqueness of the complete key.
        # Supports both single-column and composite keys.

        if primary_key:

            duplicate_key_groups = (
                dataframe
                .groupBy(*primary_key)
                .count()
                .filter(F.col("count") > 1)
                .count()
            )

            primary_key_duplicate_count = (
                duplicate_key_groups
            )

            print(
                "Duplicate primary-key groups: "
                f"{primary_key_duplicate_count:,}"
            )


        if primary_key_null_count > 0:
            raise ValueError(
                "Primary key contains NULL values"
            )

        if primary_key_duplicate_count > 0:
            raise ValueError(
                "Duplicate primary keys found"
            )


        # ----------------------------------------------------
        # Step 4.5: Add ingestion metadata
        # ----------------------------------------------------

        dataframe = dataframe.withColumns({
            "source_file": F.lit(source_file),
            "ingested_at": F.current_timestamp(),
            "run_id": F.lit(run_id)
        })

        print("Ingestion metadata added")


        # ----------------------------------------------------
        # Step 4.6: Validate metadata
        # ----------------------------------------------------

        metadata_columns = [
            "source_file",
            "ingested_at",
            "run_id"
        ]

        metadata_null_counts = dataframe.agg(
            *[
                F.sum(
                    F.when(
                        F.col(column).isNull(), 1
                    ).otherwise(0)
                ).alias(column)
                for column in metadata_columns
            ]
        ).first().asDict()

        metadata_validation_passed = all(
            count == 0
            for count in metadata_null_counts.values()
        )

        if not metadata_validation_passed:
            raise ValueError(
                "Metadata validation failed"
            )

        print("Metadata validation passed")


        # ----------------------------------------------------
        # Step 4.7: Write to Bronze Delta table
        # ----------------------------------------------------

        (
            dataframe.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        print(f"Data written to {target_table}")


        # ----------------------------------------------------
        # Step 4.8: Verify the Bronze table
        # ----------------------------------------------------

        bronze_dataframe = spark.table(target_table)

        bronze_row_count = bronze_dataframe.count()

        bronze_column_count = len(
            bronze_dataframe.columns
        )

        print(
            f"Bronze rows: {bronze_row_count:,}"
        )

        print(
            f"Bronze columns: {bronze_column_count}"
        )


        if bronze_row_count != row_count:
            raise ValueError(
                "Bronze row count does not match source"
            )

        if bronze_column_count != column_count + 3:
            raise ValueError(
                "Bronze column count is incorrect"
            )


        # ----------------------------------------------------
        # Step 4.9: Return successful result
        # ----------------------------------------------------

        print(
            f"Ingestion completed successfully: {dataset_name}"
        )

        return {
            "dataset": dataset_name,
            "source_rows": row_count,
            "bronze_rows": bronze_row_count,
            "bronze_columns": bronze_column_count,
            "status": "SUCCESS",
            "run_id": run_id,
            "error": ""
        }


    except Exception as error:

        print(
            f"Ingestion failed for {dataset_name}: {str(error)}"
        )

        return {
            "dataset": dataset_name,
            "source_rows": None,
            "bronze_rows": None,
            "bronze_columns": None,
            "status": "FAILED",
            "run_id": run_id,
            "error": str(error)[:1000]
        }


# ============================================================
# Step 5: Run the pipeline for all nine datasets
# ============================================================

ingestion_results = []

for dataset_name, dataset_config in datasets.items():

    result = ingest_dataset(
        dataset_name,
        dataset_config
    )

    ingestion_results.append(result)


# ============================================================
# Step 6: Display the ingestion summary
# ============================================================

results_df = spark.createDataFrame(
    ingestion_results
)

display(results_df)


# ============================================================
# Step 7: Print overall pipeline status
# ============================================================

successful_datasets = sum(
    1
    for result in ingestion_results
    if result["status"] == "SUCCESS"
)

failed_datasets = sum(
    1
    for result in ingestion_results
    if result["status"] == "FAILED"
)

print("\n" + "=" * 60)
print("BRONZE INGESTION PIPELINE SUMMARY")
print("=" * 60)

print(f"Total datasets: {len(datasets)}")
print(f"Successful datasets: {successful_datasets}")
print(f"Failed datasets: {failed_datasets}")

if failed_datasets == 0:
    print("All datasets ingested successfully")
else:
    print(
        "Some datasets failed. "
        "Check the result table for details."
    )
